LOAD LIBRARIES

In [ ]:
import oracle_repo as orep
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from torch.utils.data import Dataset, DataLoader
from sentence_transformers import SentenceTransformer, util

PRECONFIGURE THE PARAMETERS

In [ ]:
responses = [] #list of responses of n different models at t.

# Example responses
responses = [
    {"model_type": "gpt-4", "agent_type": "LLM", "message": "Action 1: Do this..."},
    {"model_type": "gemini", "agent_type": "LLM", "message": "Action 2: Do that..."},
    {"model_type": "mistral", "agent_type": "LLM", "message": "Action 3: Do something else..."}
]

In [ ]:
#Initialize parameters
adaptation_rate = 0 #initialize adaptation rate.
num_agents = int() #should be an int object that generates itself based on number of agents n.
messages = [] #cache to store list of best_responses for each request iteration t.
t = 0 # initialize time.

In [ ]:
# Define the problem type
# Request user input on problem type
problem_types = ["deterministic", "predictive", "code_based", "philosophical"]

print("Choose problem type:")
for p_type in problem_types:
    print(f"- {p_type}")

# Get user input for problem type
problem_type = input("Enter the problem type: ").strip().lower()

if problem_type not in problem_types:
    raise ValueError("Invalid problem_type provided")

In [ ]:
problem_scope = ["open", "closed"]

print("Choose problem scope:")
for p_type in problem_scope:
    print(f"- {p_type}")

# Get user input for problem scope
problem_scope = input("Enter the problem scope (open or closed problem?): ").strip().lower()

if problem_scope not in problem_scope:
    raise ValueError("Invalid problem_scope provided")


GENERATE TRAINING PARAMETERS

In [ ]:
# Calculate fitness for each response
fitness_scores = []
for response in responses:
    text = response["message"]
    ag_rationality, model_type_label, agent_type_label = orep.evaluate_fitness(response["model_type"], response["agent_type"], problem_type)
    fitness_scores.append(ag_rationality)

# Calculate consensus score
consensus_score = orep.calculate_consensus_score(responses)

# Calculate the action maximization score for each response
act_max_scores = []
for fitness in fitness_scores:
    ag_rationality = fitness[0]
    act_max_score = orep.calculate_act_max_score(adaptation_rate, consensus_score, ag_rationality)
    act_max_scores.append(act_max_score)

# # Output the scores
# for i, response in enumerate(responses):
#     print(f"Response from {response['model_type']} - Agent Rationality: {fitness_scores[i][0]}, Act Max Score: {act_max_scores[i]}")

# Select the best response
best_response_index = act_max_scores.index(max(act_max_scores))
best_response = responses[best_response_index]
model_type = best_response["model_type"]
act_max_score = max(act_max_scores)

messages.append(best_response)

In [ ]:
error_rate = orep.evaluate_error_rate(messages)
problem_scope_label, problem_type_label, time, num_agents, num_steps_proposed, ratio_steps_left = orep.evaluate_environment(problem_scope, problem_type, time, num_agents, response)

In [ ]:
#generate input vector

orep.generate_feature_vector(act_max_score, model_type, agent_type=None, problem_scope_label, problem_type_label, time=t, num_agents, response, adaptation_rate=0)


DEFINE THE NETWORK

In [ ]:
class LSTMNetwork(nn.Module):
    def __init__(self, input_size, hidden_size):
        super(LSTMNetwork, self).__init__()
        self.hidden_size = hidden_size
        
        self.lstm = nn.LSTM(input_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, 1)
        self.sigmoid = nn.Sigmoid()
        
    def forward(self, x):
        h0 = torch.zeros(1, x.size(0), self.hidden_size).to(x.device)
        c0 = torch.zeros(1, x.size(0), self.hidden_size).to(x.device)
        
        out, _ = self.lstm(x, (h0, c0))
        out = out[:, -1, :]  # Get the last time step output
        out = self.fc(out)
        out = self.sigmoid(out)
        return out

class FeatureDataset(Dataset):
    def __init__(self, feature_vectors, labels):
        self.feature_vectors = feature_vectors
        self.labels = labels
    
    def __len__(self):
        return len(self.feature_vectors)
    
    def __getitem__(self, idx):
        return torch.tensor(self.feature_vectors[idx], dtype=torch.float32), torch.tensor(self.labels[idx], dtype=torch.float32)

# Example feature vectors and labels
feature_vectors = [orep.generate_feature_vector(env_params, responses, api_response_text, rationality_params, accuracy, benchmark_performance_score, messages) for _ in range(100)]
labels = np.random.randint(0, 2, size=100)  # Binary labels: 1 for success, 0 for failure

# Create dataset and dataloader
dataset = FeatureDataset(feature_vectors, labels)
dataloader = DataLoader(dataset, batch_size=10, shuffle=True)

# Hyperparameters
input_size = len(feature_vectors[0])  # Number of features in the feature vector
hidden_size = 15

# Initialize the network, loss function, and optimizer
model = LSTMNetwork(input_size, hidden_size)
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

TRAIN THE NETWORK

In [ ]:
# Training loop
num_epochs = 20
for epoch in range(num_epochs):
    for i, (features, labels) in enumerate(dataloader):
        # Reshape input to (batch_size, seq_length, input_size)
        features = features.unsqueeze(1)
        
        # Forward pass
        outputs = model(features)
        loss = criterion(outputs.squeeze(), labels)
        
        # Backward pass and optimization
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    
    print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}')

print('Training completed.')

PREDICT AND EVALUATE SUCCESS

In [ ]:
def predict_success(model, feature_vector, threshold=0.55):
    model.eval()
    feature_tensor = torch.tensor(feature_vector, dtype=torch.float32).unsqueeze(0).unsqueeze(0)  # Shape (1, 1, input_size)
    with torch.no_grad():
        probability = model(feature_tensor).item()
    return probability, probability > threshold

# Example usage
test_feature_vector = generate_feature_vector(env_params, responses, api_response_text, rationality_params, accuracy, benchmark_performance_score, messages)
probability, is_success = predict_success(model, test_feature_vector)
print(f'Probability of Success: {probability:.4f}, Success: {is_success}')